# Guardrailer Embedding Model Evaluation
## Lightweight Embedding Models: Centroid Similarity, Linear Probes, and AUC-ROC

**Runtime:** Enable GPU (T4) in Kaggle Settings → Accelerator → GPU T4 x2
**Checkpointing:** All progress auto-saves to /kaggle/working/checkpoints/
**Resume:** If interrupted, re-run all cells — notebook resumes from last checkpoint

In [ ]:
!pip install -q sentence-transformers scikit-learn pandas numpy matplotlib seaborn

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"GPU Memory: {props.total_memory / 1e9:.1f} GB")
    print(f"Compute Capability: {props.major}.{props.minor}")

CUDA_WORKS = False
if torch.cuda.is_available():
    try:
        _t = torch.randn(10).cuda()
        _t = _t * 2
        del _t
        torch.cuda.empty_cache()
        CUDA_WORKS = True
        print("CUDA kernel test: PASSED")
    except Exception as e:
        print(f"CUDA kernel test: FAILED ({e})")
else:
    print("No GPU detected.")

DEVICE = "cuda" if CUDA_WORKS else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, confusion_matrix, roc_curve
from sentence_transformers import SentenceTransformer

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 150

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
OUTPUT_DIR = "/kaggle/working"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

class CheckpointManager:
    def __init__(self, checkpoint_dir=CHECKPOINT_DIR):
        self.checkpoint_dir = checkpoint_dir
        self.manifest_path = os.path.join(checkpoint_dir, "manifest.json")
        self.manifest = self._load_manifest()

    @staticmethod
    def _safe_key(name):
        return name.replace("/", "_")

    def _load_manifest(self):
        if os.path.exists(self.manifest_path):
            with open(self.manifest_path, "r") as f:
                return json.load(f)
        return {"completed_models": [], "saved_embeddings": {}}

    def _save_manifest(self):
        with open(self.manifest_path, "w") as f:
            json.dump(self.manifest, f, indent=2)

    def is_model_done(self, model_name):
        return model_name in self.manifest["completed_models"]

    def save_embeddings(self, model_name, split, embeddings, labels):
        key = f"{self._safe_key(model_name)}_{split}"
        emb_path = os.path.join(self.checkpoint_dir, f"{key}_embeddings.npy")
        label_path = os.path.join(self.checkpoint_dir, f"{key}_labels.npy")
        np.save(emb_path, embeddings)
        np.save(label_path, labels)
        self.manifest["saved_embeddings"][key] = {
            "path": emb_path,
            "label_path": label_path,
            "shape": list(embeddings.shape),
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        }
        self._save_manifest()
        print(f"  Checkpoint saved: {key} ({embeddings.shape[0]} samples, {embeddings.shape[1]} dims)")

    def load_embeddings(self, model_name, split):
        key = f"{self._safe_key(model_name)}_{split}"
        if key in self.manifest["saved_embeddings"]:
            info = self.manifest["saved_embeddings"][key]
            if os.path.exists(info["path"]) and os.path.exists(info["label_path"]):
                embeddings = np.load(info["path"])
                labels = np.load(info["label_path"])
                print(f"  Loaded checkpoint: {key} ({embeddings.shape[0]} samples)")
                return embeddings, labels
        return None, None

    def mark_model_done(self, model_name, results):
        self.manifest["completed_models"].append(model_name)
        self.manifest[f"results_{model_name}"] = results
        self._save_manifest()
        print(f"  Model marked complete: {model_name}")

    def get_results(self, model_name):
        return self.manifest.get(f"results_{model_name}", None)

    def save_proba(self, model_name, proba, y_test):
        key = self._safe_key(model_name)
        path = os.path.join(self.checkpoint_dir, f"{key}_proba.npy")
        label_path = os.path.join(self.checkpoint_dir, f"{key}_ytest.npy")
        np.save(path, proba)
        np.save(label_path, y_test)

    def load_proba(self, model_name):
        key = self._safe_key(model_name)
        path = os.path.join(self.checkpoint_dir, f"{key}_proba.npy")
        label_path = os.path.join(self.checkpoint_dir, f"{key}_ytest.npy")
        if os.path.exists(path) and os.path.exists(label_path):
            proba = np.load(path)
            y_test = np.load(label_path)
            print(f"  Loaded cached proba: {key}")
            return proba, y_test, None
        return None

    def get_status(self):
        print(f"\n{'='*50}")
        print(f"CHECKPOINT STATUS")
        print(f"{'='*50}")
        print(f"Completed models: {self.manifest['completed_models']}")
        print(f"Saved embeddings: {list(self.manifest['saved_embeddings'].keys())}")
        print(f"{'='*50}\n")

ckpt = CheckpointManager()
ckpt.get_status()

In [ ]:
DATA_PATH = "/kaggle/input/datasets/prashannadeveloper/guardrailer-dataset-v1/guardrailer_dataset_v1.parquet"

df = pd.read_parquet(DATA_PATH)
df["text"] = df["prompt_text"]
df["label"] = df["is_malicious"].astype(int)

print(f"Total samples: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nLabel distribution:")
print(df["label"].value_counts())
print(f"\nLabel distribution (normalized):")
print(df["label"].value_counts(normalize=True))

print("\n--- Sample malicious text ---")
print(df[df["label"] == 1]["text"].iloc[0][:200])
print("\n--- Sample benign text ---")
print(df[df["label"] == 0]["text"].iloc[0][:200])

In [ ]:
print(f"Full dataset: {len(df)} samples")
print(f"Class distribution:\n{df['label'].value_counts(normalize=True)}")

sampled_full, _ = train_test_split(
    df, train_size=30000, random_state=42, stratify=df["label"]
)
print(f"\nSampled pool: {len(sampled_full)} samples")
print(f"Class distribution:\n{sampled_full['label'].value_counts(normalize=True)}")

subset_pool, subset_c = train_test_split(
    sampled_full, test_size=10000, random_state=42, stratify=sampled_full["label"]
)

subset_a, subset_b = train_test_split(
    subset_pool, test_size=10000, random_state=42, stratify=subset_pool["label"]
)

print(f"\nSubset A (MiniLM):    {len(subset_a)} samples")
print(f"  Class distribution: {subset_a['label'].value_counts(normalize=True).to_dict()}")
print(f"\nSubset B (e5-small):  {len(subset_b)} samples")
print(f"  Class distribution: {subset_b['label'].value_counts(normalize=True).to_dict()}")
print(f"\nSubset C (bge-small): {len(subset_c)} samples")
print(f"  Class distribution: {subset_c['label'].value_counts(normalize=True).to_dict()}")

assert len(set(subset_a.index) & set(subset_b.index)) == 0, "A and B overlap!"
assert len(set(subset_a.index) & set(subset_c.index)) == 0, "A and C overlap!"
assert len(set(subset_b.index) & set(subset_c.index)) == 0, "B and C overlap!"
print("\nAll subsets are mutually exclusive")

model_assignments = {
    "sentence-transformers/all-MiniLM-L6-v2": subset_a,
    "intfloat/e5-small-v2": subset_b,
    "BAAI/bge-small-en-v1.5": subset_c,
}

subset_a.to_csv("/kaggle/working/subset_a_minilm.csv", index=False)
subset_b.to_csv("/kaggle/working/subset_b_e5small.csv", index=False)
subset_c.to_csv("/kaggle/working/subset_c_bgesmall.csv", index=False)
print("\nSubsets saved to /kaggle/working/")

In [ ]:
def evaluate_embedding(name, model, X_train_text, X_test_text, y_train, y_test, ckpt_manager, device="cpu"):
    print(f"\n{'='*60}")
    print(f"Evaluating: {name}")
    print(f"Device: {device}")
    print(f"{'='*60}")

    train_emb, train_labels = ckpt_manager.load_embeddings(name, "train")
    test_emb, test_labels = ckpt_manager.load_embeddings(name, "test")

    # Try to load cached proba/y_test from checkpoint
    cached = ckpt_manager.load_proba(name)

    if train_emb is not None and test_emb is not None and cached is not None:
        print("  Using checkpointed embeddings + proba (skipping encoding + probe)")
        encode_time = 0
        proba, test_labels_loaded, results_meta = cached
        test_labels = test_labels_loaded
    elif train_emb is not None and test_emb is not None:
        print("  Using checkpointed embeddings (skipping encoding)")
        encode_time = 0
        proba = None
    else:
        start = time.time()
        print(f"  Encoding train set on {device}...")
        train_emb = model.encode(
            X_train_text, batch_size=128, show_progress_bar=True,
            normalize_embeddings=True, device=device
        )
        train_emb = np.array(train_emb)
        train_labels = np.array(y_train)
        ckpt_manager.save_embeddings(name, "train", train_emb, train_labels)

        print(f"  Encoding test set on {device}...")
        test_emb = model.encode(
            X_test_text, batch_size=128, show_progress_bar=True,
            normalize_embeddings=True, device=device
        )
        test_emb = np.array(test_emb)
        test_labels = np.array(y_test)
        ckpt_manager.save_embeddings(name, "test", test_emb, test_labels)

        encode_time = time.time() - start
        print(f"  Encoding time: {encode_time:.1f}s")
        proba = None

    # Centroid similarity
    centroid_safe = train_emb[train_labels == 0].mean(axis=0)
    centroid_malicious = train_emb[train_labels == 1].mean(axis=0)
    centroid_similarity = np.dot(centroid_safe, centroid_malicious)

    test_safe_emb = test_emb[test_labels == 0]
    test_mal_emb = test_emb[test_labels == 1]
    test_centroid_sim = np.dot(test_safe_emb.mean(axis=0), test_mal_emb.mean(axis=0))

    safe_to_centroid = np.mean([np.dot(e, centroid_safe) for e in test_safe_emb])
    mal_to_centroid = np.mean([np.dot(e, centroid_malicious) for e in test_mal_emb])

    print(f"\n  Centroid Similarity (train): {centroid_similarity:.6f}")
    print(f"  Centroid Similarity (test):  {test_centroid_sim:.6f}")

    # Linear probe (skip if proba already loaded from checkpoint)
    if proba is None:
        print("  Training linear probe...")
        probe = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
        probe.fit(train_emb, train_labels)
        preds = probe.predict(test_emb)
        proba = probe.predict_proba(test_emb)[:, 1]
    else:
        print("  Using checkpointed probe results")
        preds = (proba >= 0.5).astype(int)

    acc = accuracy_score(test_labels, preds)
    f1 = f1_score(test_labels, preds)
    prec = precision_score(test_labels, preds)
    rec = recall_score(test_labels, preds)
    auc = roc_auc_score(test_labels, proba)
    cm = confusion_matrix(test_labels, preds)

    print(f"\n  Linear Probe Results:")
    print(f"    Accuracy:  {acc:.4f}")
    print(f"    F1-Score:  {f1:.4f}")
    print(f"    Precision: {prec:.4f}")
    print(f"    Recall:    {rec:.4f}")
    print(f"    AUC-ROC:   {auc:.4f}")

    # Cache proba + y_test in checkpoint for plot_results after resume
    ckpt_manager.save_proba(name, proba, test_labels)

    results = {
        "model_name": name,
        "centroid_similarity_train": float(centroid_similarity),
        "centroid_similarity_test": float(test_centroid_sim),
        "safe_to_centroid": float(safe_to_centroid),
        "malicious_to_centroid": float(mal_to_centroid),
        "accuracy": float(acc),
        "f1": float(f1),
        "precision": float(prec),
        "recall": float(rec),
        "auc_roc": float(auc),
        "confusion_matrix": cm.tolist(),
        "encode_time_sec": float(encode_time),
        "test_proba": proba,
        "y_test": test_labels,
    }

    ckpt_manager.mark_model_done(name, {k: v for k, v in results.items()
                                         if k not in ["test_proba", "y_test"]})

    import torch as _torch
    if _torch.cuda.is_available():
        _torch.cuda.empty_cache()

    return results

In [ ]:
def plot_results(results_list):
    output_dir = "/kaggle/working/"
    os.makedirs(output_dir, exist_ok=True)

    # Figure 1: Centroid Similarity
    fig, ax = plt.subplots(figsize=(8, 5))
    names = [r["model_name"].split("/")[-1] for r in results_list]
    train_sims = [r["centroid_similarity_train"] for r in results_list]
    test_sims = [r["centroid_similarity_test"] for r in results_list]
    x = np.arange(len(names))
    width = 0.35
    bars1 = ax.bar(x - width/2, train_sims, width, label="Train Centroids", color="#2196F3", alpha=0.8)
    bars2 = ax.bar(x + width/2, test_sims, width, label="Test Centroids", color="#FF5722", alpha=0.8)
    ax.axhline(y=0.85, color="green", linestyle="--", alpha=0.5, label="Threshold (0.85)")
    ax.set_ylabel("Cosine Similarity")
    ax.set_title("Inter-Class Centroid Similarity by Embedding Model")
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=15)
    ax.legend()
    ax.set_ylim(0.5, 1.05)
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                f"{bar.get_height():.4f}", ha="center", va="bottom", fontsize=9)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                f"{bar.get_height():.4f}", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plt.savefig(output_dir + "fig_centroid_similarity.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Figure 2: Linear Probe Performance
    fig, ax = plt.subplots(figsize=(8, 5))
    metrics = ["accuracy", "f1", "precision", "recall", "auc_roc"]
    metric_labels = ["Accuracy", "F1-Score", "Precision", "Recall", "AUC-ROC"]
    x = np.arange(len(names))
    width = 0.15
    colors = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0", "#F44336"]
    for i, (metric, label, color) in enumerate(zip(metrics, metric_labels, colors)):
        values = [r[metric] for r in results_list]
        ax.bar(x + i * width - 2*width, values, width, label=label, color=color, alpha=0.8)
    ax.set_ylabel("Score")
    ax.set_title("Linear Probe Performance by Embedding Model")
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=15)
    ax.legend(loc="lower right")
    ax.set_ylim(0.4, 1.05)
    plt.tight_layout()
    plt.savefig(output_dir + "fig_linear_probe_performance.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Figure 3: ROC Curves
    fig, ax = plt.subplots(figsize=(8, 6))
    colors_roc = ["#2196F3", "#4CAF50", "#FF9800"]
    for r, color in zip(results_list, colors_roc):
        fpr, tpr, _ = roc_curve(r["y_test"], r["test_proba"])
        ax.plot(fpr, tpr, color=color, linewidth=2,
                label=f'{r["model_name"].split("/")[-1]} (AUC={r["auc_roc"]:.3f})')
    ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random (AUC=0.500)")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curves: Linear Probe on Embedding Space")
    ax.legend(loc="lower right")
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])
    plt.tight_layout()
    plt.savefig(output_dir + "fig_roc_curves.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Figure 4: Score Distribution
    best = max(results_list, key=lambda r: r["auc_roc"])
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    safe_scores = best["test_proba"][best["y_test"] == 0]
    mal_scores = best["test_proba"][best["y_test"] == 1]
    axes[0].hist(safe_scores, bins=50, alpha=0.7, color="#4CAF50", label="Safe", density=True)
    axes[0].hist(mal_scores, bins=50, alpha=0.7, color="#F44336", label="Malicious", density=True)
    axes[0].set_xlabel("Predicted Probability (Malicious)")
    axes[0].set_ylabel("Density")
    axes[0].set_title(f"Score Distribution: {best['model_name'].split('/')[-1]}")
    axes[0].legend()
    margins_dist = best["test_proba"][best["y_test"] == 1] - best["test_proba"][best["y_test"] == 0].mean()
    axes[1].hist(margins_dist, bins=50, color="#FF9800", alpha=0.7, edgecolor="black")
    axes[1].axvline(x=0, color="red", linestyle="--", linewidth=2, label="Decision boundary")
    axes[1].set_xlabel("Score Margin (relative to safe centroid)")
    axes[1].set_ylabel("Count")
    axes[1].set_title(f"Margin Distribution: {best['model_name'].split('/')[-1]}")
    axes[1].legend()
    plt.tight_layout()
    plt.savefig(output_dir + "fig_score_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Figure 5: Comparison Table
    fig, ax = plt.subplots(figsize=(10, 4))
    table_data = []
    for r in results_list:
        table_data.append([
            f"{r['accuracy']:.4f}", f"{r['f1']:.4f}", f"{r['precision']:.4f}",
            f"{r['recall']:.4f}", f"{r['auc_roc']:.4f}", f"{r['centroid_similarity_test']:.4f}",
        ])
    row_labels = [r["model_name"].split("/")[-1] for r in results_list]
    col_labels = ["Accuracy", "F1", "Precision", "Recall", "AUC-ROC", "Centroid Sim"]
    table = ax.table(cellText=table_data, rowLabels=row_labels, colLabels=col_labels,
                     loc="center", cellLoc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1.2, 1.8)
    ax.axis("off")
    ax.set_title("Embedding Model Comparison Summary", fontsize=14, pad=20)
    plt.tight_layout()
    plt.savefig(output_dir + "fig_comparison_table.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
all_results = []

for name, subset in model_assignments.items():
    if ckpt.is_model_done(name):
        print(f"\nSKIPPING: {name} (already completed, loaded from checkpoint)")
        saved_results = ckpt.get_results(name)
        # Load cached proba and y_test for plotting
        cached = ckpt.load_proba(name)
        if cached is not None:
            saved_results["test_proba"] = cached[0]
            saved_results["y_test"] = cached[1]
        else:
            saved_results["test_proba"] = None
            saved_results["y_test"] = None
        all_results.append(saved_results)
        continue

    print(f"\n{'#'*60}")
    print(f"PROCESSING: {name}")
    print(f"SUBSET: {subset.shape[0]} samples")
    print(f"{'#'*60}")

    sub_train, sub_test = train_test_split(
        subset, test_size=0.20, random_state=42, stratify=subset["label"]
    )

    X_tr = sub_train["text"].astype(str).tolist()
    y_tr = sub_train["label"].values
    X_te = sub_test["text"].astype(str).tolist()
    y_te = sub_test["label"].values

    print(f"  Train: {len(X_tr)}  Test: {len(X_te)}")

    model = SentenceTransformer(name)
    print(f"  Model loaded. Using device: {DEVICE}")

    result = evaluate_embedding(name, model, X_tr, X_te, y_tr, y_te, ckpt, device=DEVICE)

    del model
    import torch as _torch
    if _torch.cuda.is_available():
        _torch.cuda.empty_cache()

    all_results.append(result)

    intermediate_path = os.path.join(OUTPUT_DIR, "embedding_results_intermediate.json")
    with open(intermediate_path, "w") as f:
        json.dump([{k: v for k, v in r.items() if k not in ["test_proba", "y_test"]}
                    for r in all_results], f, indent=2)
    print(f"  Intermediate results saved")

ckpt.get_status()
print(f"\nAll {len(all_results)} models processed successfully!")

In [ ]:
plot_results(all_results)

In [ ]:
output_dir = "/kaggle/working/"
summary = []
for r in all_results:
    entry = {k: v for k, v in r.items() if k not in ["test_embeddings", "test_proba", "y_test"]}
    summary.append(entry)

with open(output_dir + "embedding_results_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Saved: embedding_results_summary.json")
print("\n" + json.dumps(summary, indent=2))

In [ ]:
results_df = pd.DataFrame(summary)
results_df.to_csv(output_dir + "embedding_results_detailed.csv", index=False)
print("Saved: embedding_results_detailed.csv")

print("\n--- SUMMARY TABLE FOR PAPER ---")
print(results_df[["model_name", "centroid_similarity_test", "accuracy", "f1", "auc_roc"]].to_string(index=False))

## Expected Outputs (download from /kaggle/working/)

### CSV Files
- `embedding_results_summary.json` — Full results for all models
- `embedding_results_detailed.csv` — Tabular format
- `subset_a_minilm.csv` — Subset A (MiniLM samples)
- `subset_b_e5small.csv` — Subset B (e5-small samples)
- `subset_c_bgesmall.csv` — Subset C (bge-small samples)

### Visualization PNGs
- `fig_centroid_similarity.png` — Centroid similarity comparison bar chart
- `fig_linear_probe_performance.png` — Multi-metric bar chart
- `fig_roc_curves.png` — ROC curves for all models
- `fig_score_distribution.png` — Score distribution + margin analysis (best model)
- `fig_comparison_table.png` — Summary table heatmap

### What to Report in the Paper
After downloading, fill in this table:

| Embedding Model | Subset | Samples | Centroid Sim | Linear Probe Acc | AUC-ROC | Collapse? |
|-----------------|--------|---------|-------------|-----------------|---------|----------|
| all-MiniLM-L6-v2 | A | 5,000 | ? | ? | ? | Yes if AUC < 0.55 and Centroid Sim > 0.95 |
| e5-small-v2 | B | 5,000 | ? | ? | ? | Yes if AUC < 0.55 and Centroid Sim > 0.95 |
| bge-small-en-v1.5 | C | 5,000 | ? | ? | ? | Yes if AUC < 0.55 and Centroid Sim > 0.95 |

### Subset Distribution Check
Also verify and report:

| Subset | Safe Samples | Malicious Samples | Safe % | Malicious % |
|--------|-------------|-------------------|--------|-------------|
| A (MiniLM) | ? | ? | ?% | ?% |
| B (e5-small) | ? | ? | ?% | ?% |
| C (bge-small) | ? | ? | ?% | ?% |
| Full dataset | ? | ? | ?% | ?% |